In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, Concatenate, GlobalAveragePooling2D, Lambda
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import Sequence
from sklearn.model_selection import train_test_split
import os

# Load handcrafted features
df = pd.read_csv("../Dataset/Features/signature_features.csv")
labels = df["label"].values  # 0 = Genuine, 1 = Forged

# Image paths
def get_path(filename, label):
    return f"../Dataset/Processing/AugmentedDataset/{'Genuine' if label == 0 else 'Forged'}/{filename}"
image_paths = [get_path(file, label) for file, label in zip(df["filename"], labels)]

# Handcrafted features
handcrafted_features = df.iloc[:, 2:].values  # Exclude filename & label

# Split dataset
train_img_paths, val_img_paths, train_labels, val_labels, train_features, val_features = train_test_split(
    image_paths, labels, handcrafted_features, test_size=0.2, random_state=42, stratify=labels)

# Image settings
image_size = (299, 299)
feature_dim = handcrafted_features.shape[1]

# Load InceptionV3 (CNN Feature Extractor)
base_model = InceptionV3(weights="imagenet", include_top=False, input_shape=(299, 299, 3))
cnn_output = GlobalAveragePooling2D()(base_model.output)
cnn_model = Model(inputs=base_model.input, outputs=cnn_output, name="CNN_Extractor")

# Data Generator with Hard Negative Mining
class SiameseDataGenerator(Sequence):
    def __init__(self, image_paths, labels, handcrafted_features, batch_size=32):
        self.image_paths = np.array(image_paths)
        self.labels = np.array(labels)
        self.handcrafted_features = np.array(handcrafted_features)
        self.batch_size = batch_size
        self.genuine_indices = np.where(labels == 0)[0]
        self.forged_indices = np.where(labels == 1)[0]
    
    def __len__(self):
        return int(np.floor(len(self.image_paths) / self.batch_size))
    
    def __getitem__(self, index):
        batch_anchors, batch_positives, batch_negatives, batch_labels = [], [], [], []
        for _ in range(self.batch_size):
            anchor_idx = np.random.choice(self.genuine_indices)
            positive_idx = np.random.choice(self.genuine_indices)
            negative_idx = self.hard_negative_score(anchor_idx, self.forged_indices)
            
            batch_anchors.append(self.load_image(self.image_paths[anchor_idx]))
            batch_positives.append(self.load_image(self.image_paths[positive_idx]))
            batch_negatives.append(self.load_image(self.image_paths[negative_idx]))
            batch_labels.append(1)  # 1 = Positive Pair, 0 = Negative Pair
        
        return ([np.array(batch_anchors), np.array(batch_positives), np.array(batch_negatives)], np.array(batch_labels))
    
    def hard_negative_score(self, anchor_idx, forged_indices):
        anchor_feat = self.handcrafted_features[anchor_idx]
        distances = [np.linalg.norm(anchor_feat - self.handcrafted_features[idx]) for idx in forged_indices]
        return forged_indices[np.argmin(distances)]
    
    @staticmethod
    def load_image(image_path):
        img = tf.keras.preprocessing.image.load_img(image_path, target_size=image_size)
        img = tf.keras.preprocessing.image.img_to_array(img)
        return tf.keras.applications.inception_v3.preprocess_input(img)

# Create data generators
train_generator = SiameseDataGenerator(train_img_paths, train_labels, train_features)
val_generator = SiameseDataGenerator(val_img_paths, val_labels, val_features)

# Siamese Network Model
def build_siamese_model():
    input_img = Input(shape=(299, 299, 3))
    input_feat = Input(shape=(feature_dim,))
    
    cnn_emb = cnn_model(input_img)
    feat_dense = Dense(64, activation="relu")(input_feat)
    merged = Concatenate()([cnn_emb, feat_dense])
    
    dense_out = Dense(128, activation="relu")(merged)
    dense_out = Dropout(0.3)(dense_out)
    return Model([input_img, input_feat], dense_out, name="Siamese_Branch")

siamese_branch = build_siamese_model()

# Define inputs for triplet loss training
anchor_input = Input(shape=(299, 299, 3))
pos_input = Input(shape=(299, 299, 3))
neg_input = Input(shape=(299, 299, 3))

anchor_feat = siamese_branch([anchor_input, Input(shape=(feature_dim,))])
pos_feat = siamese_branch([pos_input, Input(shape=(feature_dim,))])
neg_feat = siamese_branch([neg_input, Input(shape=(feature_dim,))])

def triplet_loss(_, embeddings):
    anchor, positive, negative = embeddings[0], embeddings[1], embeddings[2]
    pos_dist = tf.reduce_sum(tf.square(anchor - positive), axis=-1)
    neg_dist = tf.reduce_sum(tf.square(anchor - negative), axis=-1)
    loss = tf.maximum(pos_dist - neg_dist + 0.2, 0.0)  # Margin = 0.2
    return tf.reduce_mean(loss)

output = Lambda(lambda x: [x[0], x[1], x[2]], name="triplet_output")([anchor_feat, pos_feat, neg_feat])
siamese_model = Model([anchor_input, pos_input, neg_input], output)

siamese_model.compile(optimizer=Adam(learning_rate=0.0001), loss=triplet_loss)

# Train the Siamese Model
siamese_model.fit(train_generator, validation_data=val_generator, epochs=25)
siamese_model.save("../Model/siamese_signature_model.h5")

print("Siamese Network training completed!")


/Users/pasan_diksura/Documents/SoftwareDevelopment/Projects/Signature-Forgery-Detection-System/.venv/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


TypeError: `output_signature` must contain objects that are subclass of `tf.TypeSpec` but found <class 'list'> which is not.